# FinanceBench Retrieval Benchmark Audit

Notebook nay kiem tra qua trinh bien FinanceBench QA benchmark thanh retrieval benchmark.

Muc tieu:
- Doc benchmark goc `financebench_open_source.jsonl`.
- Doc retrieval benchmark da tao: `queries.jsonl`, `qrels.jsonl`, `unmatched_evidence.jsonl`, `mapping_report.json`.
- Visual so mau, so evidence moi mau, so qrels moi query, phuong phap match.
- Doi chieu benchmark goc voi benchmark retrieval de xem mapping co hop ly khong.

Luu y: FinanceBench khong co chunk ground truth. `qrels.jsonl` la ground truth suy ra bang cach map `evidence_text` / `evidence_page_num` vao chunks cua he thong.

In [ ]:
from pathlib import Path
import json
import re
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
FINANCEBENCH_PATH = ROOT / 'benchmark/financebench/data/financebench_open_source.jsonl'
CHUNKS_PATH = ROOT / 'outputs/financebench_eval_bge/chunks.jsonl'
QRELS_DIR = ROOT / 'outputs/financebench_eval_bge/qrels'
QUERIES_PATH = QRELS_DIR / 'queries.jsonl'
QRELS_PATH = QRELS_DIR / 'qrels.jsonl'
UNMATCHED_PATH = QRELS_DIR / 'unmatched_evidence.jsonl'
MAPPING_REPORT_PATH = QRELS_DIR / 'mapping_report.json'

pd.set_option('display.max_colwidth', 160)

In [ ]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / 'benchmark/financebench/data/financebench_open_source.jsonl').exists():
            return candidate
    raise FileNotFoundError('Cannot find project root from current working directory')

def load_jsonl(path: Path) -> list[dict]:
    if not path.exists() or path.stat().st_size == 0:
        return []
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def count_jsonl(path: Path) -> int:
    if not path.exists() or path.stat().st_size == 0:
        return 0
    with path.open('r', encoding='utf-8') as f:
        return sum(1 for line in f if line.strip())

def load_chunks_by_ids(path: Path, wanted_ids: set[str]) -> list[dict]:
    # Stream chunks.jsonl and keep only chunks referenced by qrels.
    # This avoids loading the full semantic chunk file into notebook memory.
    if not path.exists() or not wanted_ids:
        return []
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            cid = row.get('chunk_id') or row.get('id')
            if cid in wanted_ids:
                rows.append(row)
    return rows

def norm_text(text: str) -> str:
    text = str(text or '').lower().replace('\u00a0', ' ')
    text = text.replace('???', '-').replace('?', '-').replace('?', '-')
    text = re.sub(r'[^a-z0-9$%.\-()]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def doc_key(value) -> str:
    if value is None:
        return ''
    return Path(str(value).replace('\\', '/')).stem.lower()

def chunk_text(row: dict) -> str:
    return str(row.get('text') or row.get('summary') or '')

def chunk_doc(row: dict) -> str:
    for key in ['doc_name', 'source_doc', 'source_pdf', 'source_html']:
        if row.get(key):
            return doc_key(row.get(key))
    return ''

def page_range(row: dict):
    start = row.get('page_start', row.get('page'))
    end = row.get('page_end', row.get('page'))
    try:
        start = int(start) if start is not None else None
        end = int(end) if end is not None else start
    except (TypeError, ValueError):
        return None, None
    return start, end

def page_overlap(row: dict, page) -> bool:
    if page is None:
        return False
    start, end = page_range(row)
    return start is not None and end is not None and start <= int(page) <= end

def token_recall(needle: str, haystack: str) -> float:
    n = set(re.findall(r'[a-z0-9$%.\-()]+', norm_text(needle)))
    h = set(re.findall(r'[a-z0-9$%.\-()]+', norm_text(haystack)))
    return len(n & h) / len(n) if n else 0.0

In [ ]:
ROOT = find_project_root(Path.cwd())
FINANCEBENCH_PATH = ROOT / 'benchmark/financebench/data/financebench_open_source.jsonl'
CHUNKS_PATH = ROOT / 'outputs/financebench_eval_bge/chunks.jsonl'
QRELS_DIR = ROOT / 'outputs/financebench_eval_bge/qrels'
QUERIES_PATH = QRELS_DIR / 'queries.jsonl'
QRELS_PATH = QRELS_DIR / 'qrels.jsonl'
UNMATCHED_PATH = QRELS_DIR / 'unmatched_evidence.jsonl'
MAPPING_REPORT_PATH = QRELS_DIR / 'mapping_report.json'

samples = load_jsonl(FINANCEBENCH_PATH)
queries = load_jsonl(QUERIES_PATH)
qrels = load_jsonl(QRELS_PATH)
unmatched = load_jsonl(UNMATCHED_PATH)
mapping_report = json.loads(MAPPING_REPORT_PATH.read_text(encoding='utf-8')) if MAPPING_REPORT_PATH.exists() else {}

wanted_chunk_ids = {row['chunk_id'] for row in qrels}
chunks = load_chunks_by_ids(CHUNKS_PATH, wanted_chunk_ids)
chunk_file_count = mapping_report.get('chunks') or count_jsonl(CHUNKS_PATH)

samples_df = pd.DataFrame(samples)
chunks_df = pd.DataFrame(chunks)
queries_df = pd.DataFrame(queries)
qrels_df = pd.DataFrame(qrels)
unmatched_df = pd.DataFrame(unmatched)

print('Project root:', ROOT)
print('Full chunk file rows:', chunk_file_count)
print('Loaded qrel chunks:', len(chunks), '/', len(wanted_chunk_ids))
mapping_report

## 1. Benchmark goc: so mau va evidence moi mau

In [ ]:
evidence_rows = []
for sample in samples:
    qid = str(sample['financebench_id'])
    for evidence_index, ev in enumerate(sample.get('evidence') or []):
        evidence_rows.append({
            'query_id': qid,
            'evidence_index': evidence_index,
            'doc_name': sample.get('doc_name'),
            'evidence_doc_name': doc_key(ev.get('evidence_doc_name') or ev.get('doc_name') or sample.get('doc_name')),
            'evidence_page_original': ev.get('evidence_page_num'),
            'evidence_page_one_indexed': int(ev['evidence_page_num']) + 1 if ev.get('evidence_page_num') is not None else None,
            'evidence_text': ev.get('evidence_text') or '',
            'question': sample.get('question'),
        })
evidence_df = pd.DataFrame(evidence_rows)

summary = {
    'financebench_samples': len(samples_df),
    'queries': len(queries_df),
    'evidence_rows': len(evidence_df),
    'chunks_total_in_file': chunk_file_count,
    'qrel_chunks_loaded': len(chunks_df),
    'qrels': len(qrels_df),
    'unmatched_evidence': len(unmatched_df),
    'unique_docs_in_evidence': evidence_df['evidence_doc_name'].nunique(),
}
summary

In [ ]:
ev_per_query = evidence_df.groupby('query_id').size().rename('evidence_count').reset_index()
display(ev_per_query['evidence_count'].describe())

ax = ev_per_query['evidence_count'].value_counts().sort_index().plot(kind='bar', figsize=(6, 4), title='Evidence rows per FinanceBench sample')
ax.set_xlabel('evidence rows per query')
ax.set_ylabel('number of queries')
plt.show()

## 2. Doi chieu queries.jsonl voi FinanceBench goc

In [ ]:
orig_queries = samples_df[['financebench_id', 'question', 'answer', 'doc_name', 'company']].rename(columns={'financebench_id': 'query_id'})
merged_queries = orig_queries.merge(queries_df, on='query_id', how='outer', suffixes=('_orig', '_retrieval'), indicator=True)
query_checks = {
    'missing_from_queries_jsonl': int((merged_queries['_merge'] == 'left_only').sum()),
    'extra_in_queries_jsonl': int((merged_queries['_merge'] == 'right_only').sum()),
    'question_mismatch': int((merged_queries['question_orig'] != merged_queries['question_retrieval']).fillna(False).sum()),
}
query_checks

## 3. Doi chieu evidence goc voi qrels

In [ ]:
qrel_ev = qrels_df.groupby(['query_id', 'evidence_index']).agg(
    qrel_count=('chunk_id', 'count'),
    match_methods=('match_method', lambda x: ','.join(sorted(set(x)))),
).reset_index() if not qrels_df.empty else pd.DataFrame(columns=['query_id', 'evidence_index', 'qrel_count', 'match_methods'])

evidence_audit = evidence_df.merge(qrel_ev, on=['query_id', 'evidence_index'], how='left')
evidence_audit['qrel_count'] = evidence_audit['qrel_count'].fillna(0).astype(int)
evidence_audit['is_matched'] = evidence_audit['qrel_count'] > 0

coverage = {
    'evidence_rows': len(evidence_audit),
    'matched_evidence_rows': int(evidence_audit['is_matched'].sum()),
    'unmatched_evidence_rows': int((~evidence_audit['is_matched']).sum()),
    'evidence_coverage': float(evidence_audit['is_matched'].mean()) if len(evidence_audit) else None,
}
coverage

In [ ]:
display(evidence_audit['qrel_count'].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
evidence_audit['qrel_count'].value_counts().sort_index().plot(kind='bar', ax=axes[0], title='Qrels per evidence row')
axes[0].set_xlabel('qrels mapped from one evidence row')
axes[0].set_ylabel('evidence rows')
qrels_df['match_method'].value_counts().plot(kind='bar', ax=axes[1], title='Qrel match methods')
axes[1].set_xlabel('match method')
axes[1].set_ylabel('qrels')
plt.tight_layout()
plt.show()

## 4. Kiem tra qrels co dung document/page/evidence khong

In [ ]:
chunk_lookup = {row.get('chunk_id') or row.get('id'): row for row in chunks}
audit_rows = []
for row in qrels:
    chunk = chunk_lookup.get(row['chunk_id'], {})
    ev = evidence_df[(evidence_df['query_id'] == row['query_id']) & (evidence_df['evidence_index'] == row['evidence_index'])]
    ev_text = ev.iloc[0]['evidence_text'] if len(ev) else ''
    ctext = chunk_text(chunk)
    audit_rows.append({
        **row,
        'chunk_doc_name': chunk_doc(chunk),
        'doc_match': chunk_doc(chunk) == row.get('evidence_doc_name'),
        'page_match': page_overlap(chunk, row.get('evidence_page')),
        'exact_contains_now': norm_text(ev_text) in norm_text(ctext) if ev_text else False,
        'computed_token_recall': token_recall(ev_text, ctext),
        'chunk_page_start': page_range(chunk)[0],
        'chunk_page_end': page_range(chunk)[1],
        'chunk_text_preview': ctext[:300],
    })
qrel_audit_df = pd.DataFrame(audit_rows)

qrel_quality = {
    'qrels': len(qrel_audit_df),
    'doc_match_rate': float(qrel_audit_df['doc_match'].mean()) if len(qrel_audit_df) else None,
    'page_match_rate': float(qrel_audit_df['page_match'].mean()) if len(qrel_audit_df) else None,
    'exact_contains_rate': float(qrel_audit_df['exact_contains_now'].mean()) if len(qrel_audit_df) else None,
    'avg_token_recall': float(qrel_audit_df['computed_token_recall'].mean()) if len(qrel_audit_df) else None,
}
qrel_quality

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
qrel_audit_df['doc_match'].value_counts().sort_index().plot(kind='bar', ax=axes[0], title='Doc match')
qrel_audit_df['page_match'].value_counts().sort_index().plot(kind='bar', ax=axes[1], title='Page match')
qrel_audit_df['computed_token_recall'].plot(kind='hist', bins=20, ax=axes[2], title='Evidence token recall in mapped chunk')
axes[2].set_xlabel('token recall')
plt.tight_layout()
plt.show()

## 5. Moi query phu hop voi cac chunk nao

In [ ]:
query_chunk_rows = []
for row in qrels:
    chunk = chunk_lookup.get(row['chunk_id'], {})
    sample = samples_df[samples_df['financebench_id'] == row['query_id']]
    query_chunk_rows.append({
        'query_id': row['query_id'],
        'question': sample.iloc[0]['question'] if len(sample) else '',
        'answer': sample.iloc[0]['answer'] if len(sample) else '',
        'evidence_index': row.get('evidence_index'),
        'chunk_id': row['chunk_id'],
        'chunk_doc_name': chunk_doc(chunk),
        'chunk_page_start': page_range(chunk)[0],
        'chunk_page_end': page_range(chunk)[1],
        'evidence_page': row.get('evidence_page'),
        'match_method': row.get('match_method'),
        'match_score': row.get('match_score'),
        'token_recall': row.get('token_recall'),
        'sequence_ratio': row.get('sequence_ratio'),
        'chunk_text_preview': chunk_text(chunk)[:300],
    })

query_chunk_df = pd.DataFrame(query_chunk_rows)
query_chunk_df.head(20)

In [ ]:
chunks_per_query = query_chunk_df.groupby('query_id').agg(
    relevant_chunk_count=('chunk_id', 'nunique'),
    qrel_count=('chunk_id', 'count'),
    evidence_count=('evidence_index', 'nunique'),
    match_methods=('match_method', lambda x: ','.join(sorted(set(x)))),
    pages=('chunk_page_start', lambda x: ','.join(map(str, sorted(set(x.dropna().astype(int)))))),
).reset_index()
chunks_per_query = chunks_per_query.merge(queries_df[['query_id', 'question', 'answer', 'doc_name']], on='query_id', how='left')

display(chunks_per_query['relevant_chunk_count'].describe())
ax = chunks_per_query['relevant_chunk_count'].value_counts().sort_index().plot(
    kind='bar', figsize=(7, 4), title='Relevant chunks per query after evidence mapping'
)
ax.set_xlabel('unique relevant chunks per query')
ax.set_ylabel('number of queries')
plt.show()

chunks_per_query.sort_values('relevant_chunk_count', ascending=False).head(20)

Bang `query_chunk_df` ben duoi la mapping chi tiet: moi query phu hop voi chunk nao, nam o page nao, va duoc gan bang `exact_text`, `fuzzy_text` hay `page_fallback`.

In [ ]:
query_chunk_df.sort_values(['query_id', 'evidence_index', 'match_score'], ascending=[True, True, False]).head(100)

Neu muon xem rieng mot query:

In [ ]:
QUERY_ID = chunks_per_query.sort_values('relevant_chunk_count', ascending=False).iloc[0]['query_id']
print('Selected query:', QUERY_ID)
display(queries_df[queries_df['query_id'] == QUERY_ID])
display(query_chunk_df[query_chunk_df['query_id'] == QUERY_ID].sort_values(['evidence_index', 'match_score'], ascending=[True, False]))

## 6. Cac truong hop can xem lai

In [ ]:
suspicious = qrel_audit_df[
    (~qrel_audit_df['doc_match']) |
    (~qrel_audit_df['page_match']) |
    ((qrel_audit_df['match_method'] == 'fuzzy_text') & (qrel_audit_df['computed_token_recall'] < 0.60))
].copy()
suspicious[['query_id', 'chunk_id', 'match_method', 'match_score', 'doc_match', 'page_match', 'computed_token_recall', 'evidence_doc_name', 'chunk_doc_name', 'evidence_page', 'chunk_page_start', 'chunk_page_end']].head(30)

In [ ]:
if len(unmatched_df):
    display(unmatched_df.head(20))
else:
    print('No unmatched evidence rows.')

## Shell-style quick stats: qrels theo query

Cell nay thong ke nhanh tu `qrels.jsonl`: moi query relevant voi bao nhieu `chunk_id`, phan bo so chunk relevant, va top query co nhieu chunk relevant nhat.

In [ ]:
from collections import Counter, defaultdict

qrels_path = QRELS_PATH
queries_path = QUERIES_PATH

qrels_raw = load_jsonl(qrels_path)
queries_raw = load_jsonl(queries_path)
query_lookup = {row['query_id']: row for row in queries_raw}

relevant_chunks_by_query = defaultdict(set)
qrels_by_query = Counter()
methods_by_query = defaultdict(set)

for row in qrels_raw:
    qid = row['query_id']
    relevant_chunks_by_query[qid].add(row['chunk_id'])
    qrels_by_query[qid] += 1
    methods_by_query[qid].add(row.get('match_method'))

distribution = Counter(len(chunk_ids) for chunk_ids in relevant_chunks_by_query.values())

print('Total queries:', len(queries_raw))
print('Queries with qrels:', len(relevant_chunks_by_query))
print('Total qrels:', len(qrels_raw))
print('Avg relevant chunks/query:', round(sum(len(v) for v in relevant_chunks_by_query.values()) / len(queries_raw), 4))
print('\nDistribution: relevant chunk_id count -> number of queries')
for chunk_count, query_count in sorted(distribution.items()):
    print(f'{chunk_count}: {query_count}')

rows = []
for qid, chunk_ids in relevant_chunks_by_query.items():
    query = query_lookup.get(qid, {})
    rows.append({
        'query_id': qid,
        'relevant_chunk_count': len(chunk_ids),
        'qrel_count': qrels_by_query[qid],
        'match_methods': ','.join(sorted(m for m in methods_by_query[qid] if m)),
        'question': query.get('question'),
        'answer': query.get('answer'),
    })

relevant_chunks_stats_df = pd.DataFrame(rows).sort_values('relevant_chunk_count', ascending=False)
display(relevant_chunks_stats_df.head(20))